# Vave Pipeline Data Quality Validation

This notebook provides comprehensive data quality checks for the Vave geospatial risk pipeline across all layers:

* **File Upload Verification** - Validate source files and ingestion
* **Bronze Layer** - Raw data validation (counts, duplicates, schema)
* **Silver Layer** - Enriched data validation (H3 indexes, nulls, transformations)
* **Gold Layer** - Aggregated data validation (materialized views, metrics)
* **End-to-End Flow** - Cross-layer data lineage and consistency
* **Quality Metrics** - Dashboard-ready quality indicators

Run all cells to generate a complete data quality report.

---
## 1. File Upload Verification

Validate that source files are being ingested correctly into the bronze layer.

In [0]:
%sql
-- Verify files are being ingested and track ingestion metadata
SELECT 
  _metadata.file_path,
  _metadata.file_name,
  _metadata.file_size,
  _metadata.file_modification_time,
  COUNT(*) AS records_in_file,
  MIN(event_timestamp) AS earliest_event,
  MAX(event_timestamp) AS latest_event
FROM main.bronze.vave_eventsfeed
GROUP BY 
  _metadata.file_path,
  _metadata.file_name,
  _metadata.file_size,
  _metadata.file_modification_time
ORDER BY _metadata.file_modification_time DESC
LIMIT 20;

In [0]:
%sql
-- Track file ingestion over time to identify gaps
SELECT 
  DATE(_metadata.file_modification_time) AS ingestion_date,
  COUNT(DISTINCT _metadata.file_name) AS files_ingested,
  SUM(COUNT(*)) OVER (ORDER BY DATE(_metadata.file_modification_time)) AS cumulative_records
FROM main.bronze.vave_eventsfeed
GROUP BY DATE(_metadata.file_modification_time)
ORDER BY ingestion_date DESC;

In [0]:
%sql
-- Identify days with no file ingestion (potential issues)
WITH date_range AS (
  SELECT 
    DATE_ADD(CURRENT_DATE(), -seq.pos) AS check_date
  FROM (SELECT EXPLODE(SEQUENCE(0, 30)) AS pos)
),
file_dates AS (
  SELECT DISTINCT 
    DATE(_metadata.file_modification_time) AS ingestion_date
  FROM main.bronze.vave_eventsfeed
)
SELECT 
  dr.check_date,
  CASE 
    WHEN fd.ingestion_date IS NULL THEN 'MISSING'
    ELSE 'OK'
  END AS ingestion_status
FROM date_range dr
LEFT JOIN file_dates fd ON dr.check_date = fd.ingestion_date
WHERE fd.ingestion_date IS NULL
ORDER BY dr.check_date DESC;

---
## 2. Bronze Layer Data Quality

Validate raw data integrity, detect duplicates, and verify schema compliance.

In [0]:
%sql
-- Overall bronze layer statistics
SELECT 
  COUNT(*) AS total_records,
  COUNT(DISTINCT event_id) AS unique_events,
  COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_count,
  ROUND((COUNT(*) - COUNT(DISTINCT event_id)) * 100.0 / COUNT(*), 2) AS duplicate_percentage,
  MIN(event_timestamp) AS earliest_event,
  MAX(event_timestamp) AS latest_event,
  DATEDIFF(MAX(event_timestamp), MIN(event_timestamp)) AS data_span_days
FROM main.bronze.vave_eventsfeed;

In [0]:
%sql
-- Identify duplicate event_ids in bronze layer
SELECT 
  event_id,
  COUNT(*) AS occurrence_count,
  COLLECT_SET(_metadata.file_name) AS source_files,
  MIN(event_timestamp) AS first_occurrence,
  MAX(event_timestamp) AS last_occurrence
FROM main.bronze.vave_eventsfeed
GROUP BY event_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC
LIMIT 50;

In [0]:
%sql
-- Check for null values in critical fields
SELECT 
  COUNT(*) AS total_records,
  SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) AS null_event_id,
  SUM(CASE WHEN event_timestamp IS NULL THEN 1 ELSE 0 END) AS null_timestamp,
  SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) AS null_latitude,
  SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END) AS null_longitude,
  SUM(CASE WHEN risk_score IS NULL THEN 1 ELSE 0 END) AS null_risk_score,
  ROUND(SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_null_event_id,
  ROUND(SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_null_coords
FROM main.bronze.vave_eventsfeed;

In [0]:
%sql
-- Validate latitude/longitude ranges
SELECT 
  SUM(CASE WHEN latitude < -90 OR latitude > 90 THEN 1 ELSE 0 END) AS invalid_latitude,
  SUM(CASE WHEN longitude < -180 OR longitude > 180 THEN 1 ELSE 0 END) AS invalid_longitude,
  SUM(CASE WHEN latitude = 0 AND longitude = 0 THEN 1 ELSE 0 END) AS null_island_coords,
  MIN(latitude) AS min_lat,
  MAX(latitude) AS max_lat,
  MIN(longitude) AS min_lon,
  MAX(longitude) AS max_lon
FROM main.bronze.vave_eventsfeed;

In [0]:
%sql
-- Track daily ingestion patterns
SELECT 
  DATE(event_timestamp) AS event_date,
  COUNT(*) AS record_count,
  COUNT(DISTINCT event_id) AS unique_events,
  ROUND(AVG(risk_score), 2) AS avg_risk_score,
  MIN(risk_score) AS min_risk,
  MAX(risk_score) AS max_risk
FROM main.bronze.vave_eventsfeed
GROUP BY DATE(event_timestamp)
ORDER BY event_date DESC
LIMIT 30;

---
## 3. Silver Layer Data Quality

Validate enriched data with H3 geospatial indexes, transformations, and deduplication.

In [0]:
%sql
-- Silver layer statistics and comparison to bronze
WITH bronze_stats AS (
  SELECT 
    COUNT(DISTINCT event_id) AS bronze_unique_events
  FROM main.bronze.vave_eventsfeed
),
silver_stats AS (
  SELECT 
    COUNT(*) AS silver_total_records,
    COUNT(DISTINCT event_id) AS silver_unique_events,
    MIN(event_timestamp) AS earliest_event,
    MAX(event_timestamp) AS latest_event
  FROM main.silver.vave_eventsfeed
)
SELECT 
  s.*,
  b.bronze_unique_events,
  s.silver_unique_events - b.bronze_unique_events AS record_difference,
  ROUND((s.silver_unique_events - b.bronze_unique_events) * 100.0 / b.bronze_unique_events, 2) AS pct_difference
FROM silver_stats s
CROSS JOIN bronze_stats b;

In [0]:
%sql
-- Verify H3 index generation and coverage
SELECT 
  COUNT(*) AS total_records,
  SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) AS null_h3_index,
  SUM(CASE WHEN h3_ix IS NOT NULL THEN 1 ELSE 0 END) AS populated_h3_index,
  ROUND(SUM(CASE WHEN h3_ix IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS h3_coverage_pct,
  COUNT(DISTINCT h3_ix) AS unique_h3_cells,
  SUM(CASE WHEN event_dt IS NULL THEN 1 ELSE 0 END) AS null_event_dt
FROM main.silver.vave_eventsfeed;

In [0]:
%sql
-- Comprehensive null check for silver layer
SELECT 
  'event_id' AS column_name,
  SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) AS null_count,
  ROUND(SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS null_pct
FROM main.silver.vave_eventsfeed

UNION ALL

SELECT 'h3_ix', SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END), 
  ROUND(SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)
FROM main.silver.vave_eventsfeed

UNION ALL

SELECT 'event_dt', SUM(CASE WHEN event_dt IS NULL THEN 1 ELSE 0 END),
  ROUND(SUM(CASE WHEN event_dt IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)
FROM main.silver.vave_eventsfeed

UNION ALL

SELECT 'risk_score', SUM(CASE WHEN risk_score IS NULL THEN 1 ELSE 0 END),
  ROUND(SUM(CASE WHEN risk_score IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)
FROM main.silver.vave_eventsfeed

ORDER BY null_pct DESC;

In [0]:
%sql
-- Analyze event distribution across H3 cells
SELECT 
  PERCENTILE_APPROX(cell_count, 0.5) AS median_events_per_cell,
  AVG(cell_count) AS avg_events_per_cell,
  MIN(cell_count) AS min_events_per_cell,
  MAX(cell_count) AS max_events_per_cell,
  STDDEV(cell_count) AS stddev_events_per_cell
FROM (
  SELECT h3_ix, COUNT(*) AS cell_count
  FROM main.silver.vave_eventsfeed
  WHERE h3_ix IS NOT NULL
  GROUP BY h3_ix
);

In [0]:
%sql
-- Track daily silver layer processing
SELECT 
  event_dt,
  COUNT(*) AS record_count,
  COUNT(DISTINCT event_id) AS unique_events,
  COUNT(DISTINCT h3_ix) AS unique_h3_cells,
  ROUND(AVG(risk_score), 2) AS avg_risk_score,
  SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) AS h3_null_count
FROM main.silver.vave_eventsfeed
GROUP BY event_dt
ORDER BY event_dt DESC
LIMIT 30;

---
## 4. Gold Layer Verification

Validate materialized views, aggregations, and gold layer metrics.

In [0]:
%sql
-- Check materialized view metadata and refresh status
DESCRIBE EXTENDED main.gold.daily_risk_cell;

In [0]:
%sql
-- Validate daily_risk_cell aggregations
SELECT 
  COUNT(*) AS total_cell_day_records,
  COUNT(DISTINCT h3_ix) AS unique_h3_cells,
  COUNT(DISTINCT event_dt) AS unique_dates,
  MIN(event_dt) AS earliest_date,
  MAX(event_dt) AS latest_date,
  ROUND(AVG(total_events), 2) AS avg_events_per_cell_day,
  ROUND(AVG(total_risk_score), 2) AS avg_risk_per_cell_day,
  MAX(total_events) AS max_events_single_cell_day
FROM main.gold.daily_risk_cell;

In [0]:
%sql
-- Validate daily_risk_zone aggregations
SELECT 
  COUNT(*) AS total_daily_records,
  MIN(event_dt) AS earliest_date,
  MAX(event_dt) AS latest_date,
  ROUND(AVG(total_events), 2) AS avg_daily_events,
  ROUND(AVG(total_risk_score), 2) AS avg_daily_risk,
  MAX(total_events) AS max_events_single_day,
  MAX(unique_h3_cells) AS max_cells_single_day
FROM main.gold.daily_risk_zone;

In [0]:
%sql
-- Compare silver source data to gold aggregations
WITH silver_agg AS (
  SELECT 
    event_dt,
    COUNT(*) AS silver_event_count,
    COUNT(DISTINCT h3_ix) AS silver_cell_count,
    ROUND(SUM(risk_score), 2) AS silver_total_risk
  FROM main.silver.vave_eventsfeed
  GROUP BY event_dt
),
gold_agg AS (
  SELECT 
    event_dt,
    total_events AS gold_event_count,
    unique_h3_cells AS gold_cell_count,
    total_risk_score AS gold_total_risk
  FROM main.gold.daily_risk_zone
)
SELECT 
  s.event_dt,
  s.silver_event_count,
  g.gold_event_count,
  s.silver_event_count - g.gold_event_count AS event_count_diff,
  s.silver_cell_count,
  g.gold_cell_count,
  s.silver_total_risk,
  g.gold_total_risk,
  ABS(s.silver_total_risk - g.gold_total_risk) AS risk_score_diff
FROM silver_agg s
FULL OUTER JOIN gold_agg g ON s.event_dt = g.event_dt
WHERE s.silver_event_count != g.gold_event_count OR s.event_dt IS NULL OR g.event_dt IS NULL
ORDER BY s.event_dt DESC
LIMIT 20;

In [0]:
%sql
-- Verify gold layer is up-to-date with silver
WITH latest_silver AS (
  SELECT MAX(event_dt) AS latest_silver_date
  FROM main.silver.vave_eventsfeed
),
latest_gold_cell AS (
  SELECT MAX(event_dt) AS latest_gold_cell_date
  FROM main.gold.daily_risk_cell
),
latest_gold_zone AS (
  SELECT MAX(event_dt) AS latest_gold_zone_date
  FROM main.gold.daily_risk_zone
)
SELECT 
  ls.latest_silver_date,
  lgc.latest_gold_cell_date,
  lgz.latest_gold_zone_date,
  DATEDIFF(ls.latest_silver_date, lgc.latest_gold_cell_date) AS cell_lag_days,
  DATEDIFF(ls.latest_silver_date, lgz.latest_gold_zone_date) AS zone_lag_days,
  CASE 
    WHEN DATEDIFF(ls.latest_silver_date, lgc.latest_gold_cell_date) > 1 THEN 'STALE'
    ELSE 'FRESH'
  END AS cell_status,
  CASE 
    WHEN DATEDIFF(ls.latest_silver_date, lgz.latest_gold_zone_date) > 1 THEN 'STALE'
    ELSE 'FRESH'
  END AS zone_status
FROM latest_silver ls
CROSS JOIN latest_gold_cell lgc
CROSS JOIN latest_gold_zone lgz;

---
## 5. End-to-End Data Flow Validation

Validate data consistency and lineage across all pipeline layers.

In [0]:
%sql
-- Compare record counts across all layers
WITH bronze_count AS (
  SELECT 
    'Bronze' AS layer,
    COUNT(*) AS total_records,
    COUNT(DISTINCT event_id) AS unique_events
  FROM main.bronze.vave_eventsfeed
),
silver_count AS (
  SELECT 
    'Silver' AS layer,
    COUNT(*) AS total_records,
    COUNT(DISTINCT event_id) AS unique_events
  FROM main.silver.vave_eventsfeed
),
gold_cell_count AS (
  SELECT 
    'Gold (Cell)' AS layer,
    COUNT(*) AS total_records,
    SUM(total_events) AS unique_events
  FROM main.gold.daily_risk_cell
),
gold_zone_count AS (
  SELECT 
    'Gold (Zone)' AS layer,
    COUNT(*) AS total_records,
    SUM(total_events) AS unique_events
  FROM main.gold.daily_risk_zone
)
SELECT * FROM bronze_count
UNION ALL
SELECT * FROM silver_count
UNION ALL
SELECT * FROM gold_cell_count
UNION ALL
SELECT * FROM gold_zone_count;

In [0]:
%sql
-- Compare latest timestamps across all layers
SELECT 
  'Bronze' AS layer,
  MAX(event_timestamp) AS latest_timestamp,
  MAX(DATE(event_timestamp)) AS latest_date,
  DATEDIFF(CURRENT_DATE(), MAX(DATE(event_timestamp))) AS days_since_last_update
FROM main.bronze.vave_eventsfeed

UNION ALL

SELECT 
  'Silver' AS layer,
  MAX(event_timestamp) AS latest_timestamp,
  MAX(event_dt) AS latest_date,
  DATEDIFF(CURRENT_DATE(), MAX(event_dt)) AS days_since_last_update
FROM main.silver.vave_eventsfeed

UNION ALL

SELECT 
  'Gold (Cell)' AS layer,
  NULL AS latest_timestamp,
  MAX(event_dt) AS latest_date,
  DATEDIFF(CURRENT_DATE(), MAX(event_dt)) AS days_since_last_update
FROM main.gold.daily_risk_cell

UNION ALL

SELECT 
  'Gold (Zone)' AS layer,
  NULL AS latest_timestamp,
  MAX(event_dt) AS latest_date,
  DATEDIFF(CURRENT_DATE(), MAX(event_dt)) AS days_since_last_update
FROM main.gold.daily_risk_zone;

In [0]:
%sql
-- Sample event IDs and trace through all layers
WITH sample_events AS (
  SELECT DISTINCT event_id
  FROM main.silver.vave_eventsfeed
  LIMIT 10
)
SELECT 
  se.event_id,
  CASE WHEN b.event_id IS NOT NULL THEN 'YES' ELSE 'NO' END AS in_bronze,
  CASE WHEN s.event_id IS NOT NULL THEN 'YES' ELSE 'NO' END AS in_silver,
  s.h3_ix,
  s.event_dt
FROM sample_events se
LEFT JOIN main.bronze.vave_eventsfeed b ON se.event_id = b.event_id
LEFT JOIN main.silver.vave_eventsfeed s ON se.event_id = s.event_id;

---
## 6. Data Quality Metrics Dashboard

Key metrics and KPIs for monitoring pipeline health.

In [0]:
%sql
-- Comprehensive pipeline health dashboard
WITH bronze_health AS (
  SELECT 
    COUNT(*) AS bronze_records,
    COUNT(DISTINCT event_id) AS bronze_unique_events,
    MAX(event_timestamp) AS bronze_latest_timestamp
  FROM main.bronze.vave_eventsfeed
),
silver_health AS (
  SELECT 
    COUNT(*) AS silver_records,
    COUNT(DISTINCT event_id) AS silver_unique_events,
    SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) AS h3_null_count,
    MAX(event_timestamp) AS silver_latest_timestamp
  FROM main.silver.vave_eventsfeed
),
gold_health AS (
  SELECT 
    COUNT(*) AS gold_cell_records,
    MAX(event_dt) AS gold_latest_date
  FROM main.gold.daily_risk_cell
)
SELECT 
  b.bronze_records,
  b.bronze_unique_events,
  s.silver_records,
  s.silver_unique_events,
  ROUND((s.h3_null_count * 100.0 / s.silver_records), 2) AS h3_null_pct,
  g.gold_cell_records,
  DATEDIFF(CURRENT_TIMESTAMP(), b.bronze_latest_timestamp) AS bronze_age_days,
  DATEDIFF(CURRENT_TIMESTAMP(), s.silver_latest_timestamp) AS silver_age_days,
  DATEDIFF(CURRENT_DATE(), g.gold_latest_date) AS gold_age_days,
  CASE 
    WHEN DATEDIFF(CURRENT_DATE(), g.gold_latest_date) <= 1 THEN 'HEALTHY'
    WHEN DATEDIFF(CURRENT_DATE(), g.gold_latest_date) <= 3 THEN 'WARNING'
    ELSE 'CRITICAL'
  END AS pipeline_status
FROM bronze_health b
CROSS JOIN silver_health s
CROSS JOIN gold_health g;

In [0]:
%sql
-- Calculate quality scores for each layer
WITH bronze_quality AS (
  SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) AS null_ids,
    SUM(CASE WHEN latitude IS NULL OR longitude IS NULL THEN 1 ELSE 0 END) AS null_coords,
    COUNT(*) - COUNT(DISTINCT event_id) AS duplicates
  FROM main.bronze.vave_eventsfeed
),
silver_quality AS (
  SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) AS null_h3,
    SUM(CASE WHEN event_dt IS NULL THEN 1 ELSE 0 END) AS null_dates
  FROM main.silver.vave_eventsfeed
)
SELECT 
  'Bronze Layer' AS layer,
  ROUND(100 - ((bq.null_ids + bq.null_coords + bq.duplicates) * 100.0 / bq.total), 2) AS quality_score,
  CONCAT(
    'Issues: ', bq.null_ids, ' null IDs, ',
    bq.null_coords, ' null coords, ',
    bq.duplicates, ' duplicates'
  ) AS quality_details
FROM bronze_quality bq

UNION ALL

SELECT 
  'Silver Layer' AS layer,
  ROUND(100 - ((sq.null_h3 + sq.null_dates) * 100.0 / sq.total), 2) AS quality_score,
  CONCAT(
    'Issues: ', sq.null_h3, ' null H3 indexes, ',
    sq.null_dates, ' null dates'
  ) AS quality_details
FROM silver_quality sq;

In [0]:
%sql
-- Identify and rank data quality issues
WITH quality_checks AS (
  SELECT 'Bronze: Null event_id' AS issue_type, 
    SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) AS issue_count,
    'CRITICAL' AS severity
  FROM main.bronze.vave_eventsfeed
  
  UNION ALL
  
  SELECT 'Bronze: Duplicate event_id' AS issue_type,
    COUNT(*) - COUNT(DISTINCT event_id) AS issue_count,
    'HIGH' AS severity
  FROM main.bronze.vave_eventsfeed
  
  UNION ALL
  
  SELECT 'Bronze: Null coordinates' AS issue_type,
    SUM(CASE WHEN latitude IS NULL OR longitude IS NULL THEN 1 ELSE 0 END) AS issue_count,
    'HIGH' AS severity
  FROM main.bronze.vave_eventsfeed
  
  UNION ALL
  
  SELECT 'Bronze: Invalid coordinates' AS issue_type,
    SUM(CASE WHEN latitude < -90 OR latitude > 90 OR longitude < -180 OR longitude > 180 THEN 1 ELSE 0 END) AS issue_count,
    'CRITICAL' AS severity
  FROM main.bronze.vave_eventsfeed
  
  UNION ALL
  
  SELECT 'Silver: Null H3 index' AS issue_type,
    SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) AS issue_count,
    'HIGH' AS severity
  FROM main.silver.vave_eventsfeed
  
  UNION ALL
  
  SELECT 'Silver: Null event_dt' AS issue_type,
    SUM(CASE WHEN event_dt IS NULL THEN 1 ELSE 0 END) AS issue_count,
    'HIGH' AS severity
  FROM main.silver.vave_eventsfeed
)
SELECT 
  issue_type,
  issue_count,
  severity,
  CASE 
    WHEN issue_count = 0 THEN '✓ PASS'
    WHEN issue_count < 10 THEN '⚠ WARNING'
    ELSE '✗ FAIL'
  END AS status
FROM quality_checks
WHERE issue_count > 0
ORDER BY 
  CASE severity 
    WHEN 'CRITICAL' THEN 1
    WHEN 'HIGH' THEN 2
    WHEN 'MEDIUM' THEN 3
    ELSE 4
  END,
  issue_count DESC
LIMIT 10;

In [0]:
%sql
-- Track data quality metrics over time
SELECT 
  event_dt,
  COUNT(*) AS total_records,
  COUNT(DISTINCT event_id) AS unique_events,
  SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) AS h3_null_count,
  ROUND(SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS h3_null_pct,
  ROUND(AVG(risk_score), 2) AS avg_risk_score,
  CASE 
    WHEN SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) = 0 THEN 'EXCELLENT'
    WHEN SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) < 1 THEN 'GOOD'
    WHEN SUM(CASE WHEN h3_ix IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) < 5 THEN 'FAIR'
    ELSE 'POOR'
  END AS daily_quality_rating
FROM main.silver.vave_eventsfeed
GROUP BY event_dt
ORDER BY event_dt DESC
LIMIT 30;